In [11]:
# Aggregate capacity and capacity addition from the ESM by location (works for all SCMs)

from zen_garden.postprocess.results import Results
from pathlib import Path
import pandas as pd

# Configuration
dataset_name = "2020_70ts_30a"
selected_technologies = {"heat_pump", "photovoltaics", "wind_onshore"}
specific_locations = {"DE", "CH"}

base_path = Path("outputs")
results_path = base_path / dataset_name
output_dir = Path("csv_output") / dataset_name


output_dir.mkdir(parents=True, exist_ok=True)

# Load results
res = Results(results_path)
capacity = res.get_full_ts("capacity").reset_index()
capacity_addition = res.get_full_ts("capacity_addition").reset_index()

# Filter technologies
capacity = capacity[capacity["technology"].isin(selected_technologies)]
capacity_addition = capacity_addition[capacity_addition["technology"].isin(selected_technologies)]

# Save filtered raw data
capacity.round(4).to_csv(output_dir / "capacity.csv", index=False)
capacity_addition.round(4).to_csv(output_dir / "capacity_addition.csv", index=False)

# Compute ROE locations
all_locations = set(capacity["location"].unique())
roe_locations = all_locations - specific_locations

# Helper function
def aggregate_by_location(df, location_set, location_name):
    return (
        df[df["location"].isin(location_set)]
        .groupby(["technology", "capacity_type"])
        .sum(numeric_only=True)
        .assign(location=location_name)
    )

# Perform aggregation
def perform_aggregation(df):
    parts = [aggregate_by_location(df, {loc}, loc) for loc in specific_locations]
    parts.append(aggregate_by_location(df, roe_locations, "ROE"))
    return pd.concat(parts).reset_index()

capacity_aggregated = perform_aggregation(capacity)
capacity_addition_aggregated = perform_aggregation(capacity_addition)

# Save aggregated data
capacity_aggregated.round(4).to_csv(output_dir / "capacity_aggregated_by_location.csv", index=False)
capacity_addition_aggregated.round(4).to_csv(output_dir / "capacity_addition_aggregated_by_location.csv", index=False)

print(f"CSV files for dataset '{dataset_name}' saved in: {output_dir}")


CSV files for dataset '2020_70ts_30a' saved in: csv_output\2020_70ts_30a


In [12]:
# rename_year_columns.py
"""
Renames numeric year columns (e.g. "0", "1", ...) in aggregated CSVs
to actual years (e.g. "2025", "2030", ...) using system.json config.
Also reorders columns so 'location' is the third column.
"""

import json
import pandas as pd
from pathlib import Path

# CONFIGURATION
dataset_name = "2020_70ts_30a"
results_path = Path("outputs") / dataset_name
output_dir = Path("csv_output") / dataset_name

# Load system.json and construct rename map
system_path = results_path / "system.json"
with open(system_path, "r") as f:
    config = json.load(f)

ref_year = config["reference_year"]
interval = config["interval_between_years"]
n_years = config["optimized_years"]

year_map = {str(i): str(ref_year + i * interval) for i in range(n_years)}

def rename_year_columns(csv_file):
    df = pd.read_csv(csv_file)
    df.rename(columns={col: year_map[col] for col in df.columns if col in year_map}, inplace=True)

    # Reorder columns to put location third
    base_cols = ["technology", "capacity_type", "location"]
    rest = [c for c in df.columns if c not in base_cols]
    df = df[base_cols + rest]

    df.to_csv(csv_file, index=False)
    print(f"Updated: {csv_file.name}")

# Apply to both aggregated files
rename_year_columns(output_dir / "capacity_aggregated_by_location.csv")
rename_year_columns(output_dir / "capacity_addition_aggregated_by_location.csv")

Updated: capacity_aggregated_by_location.csv
Updated: capacity_addition_aggregated_by_location.csv


In [13]:
import pandas as pd
import shutil
import json
from pathlib import Path

# Configuration: define the dataset and adjust paths if needed
dataset_name = "2020_70ts_30a"
base_path = Path("./outputs")
output_dir = Path("./CSV_output") / dataset_name
esm_capacity_path = output_dir / "capacity_addition_aggregated_by_location.csv"
system_path = base_path.parent / dataset_name / "system.json"

original_dir = Path(r"C:\Users\nicol\OneDrive\Dokumente\ZEN-garden_SP\Data_PV\01_SP_MV_PV")         #path to SCM dataset
new_dir = Path(r"C:\Users\nicol\OneDrive\Dokumente\ZEN-garden_SP\Data_PV\01_SP_MV_PV_new")          #path to duplicated SCM dataset
original_dyv_path = original_dir / "set_carriers" / "pv_module" / "demand_yearly_variation.csv"     #path to demand_yearly_variation.csv (dyv)
target_dyv_path = new_dir / "set_carriers" / "pv_module" / "demand_yearly_variation.csv"            #target path to dyv file in duplicated dataset

technology = "photovoltaics"
unit_multiplier = 1000  # MW to GW to align ESM output units with PV SCM units
country_map = {"DE": "DEU", "CH": "CHE", "AT": "AUT", "NL": "NLD", "ROE": "ROE", "ROW": "ROW"}

# Clone Input Directory
if not new_dir.exists():
    shutil.copytree(original_dir, new_dir)
    print(f"Cloned input to: {new_dir}")
else:
    print(f"Directory already exists: {new_dir}")

# Load System Metadata
with open(system_path, "r") as f:
    config = json.load(f)
year_labels = [str(config["reference_year"] + i * config["interval_between_years"]) for i in range(config["optimized_years"])]

# Load Data
df_existing = pd.read_csv(original_dyv_path)
df_esm = pd.read_csv(esm_capacity_path)
df_filtered = df_esm[df_esm["technology"] == technology]

# Group and Map
df_grouped = df_filtered.groupby("location").sum(numeric_only=True).reset_index()
df_grouped["node"] = df_grouped["location"].map(country_map)
df_grouped = df_grouped[df_grouped["node"].notna()].drop(columns="location")

# Transpose and Merge
df_new = df_grouped.set_index("node").T
df_new.index.name = "year"
df_new = df_new.reset_index()
df_new["year"] = df_new["year"].astype(str)
df_existing["year"] = df_existing["year"].astype(str)

df_new = df_new.set_index("year") * unit_multiplier
df_existing = df_existing.set_index("year")

for col in df_new.columns.intersection(df_existing.columns):
    df_existing[col] = df_new[col].combine_first(df_existing[col])

# Save final file
df_result = df_existing.reset_index()
df_result.iloc[:, 1:] = df_result.iloc[:, 1:].round(3)  # Round numeric values to 3 decimal place
df_result.to_csv(target_dyv_path, index=False)
print(f"demand_yearly_variation.csv written to:\n{target_dyv_path}")


Directory already exists: C:\Users\nicol\OneDrive\Dokumente\ZEN-garden_SP\Data_PV\01_SP_MV_PV_new
demand_yearly_variation.csv written to:
C:\Users\nicol\OneDrive\Dokumente\ZEN-garden_SP\Data_PV\01_SP_MV_PV_new\set_carriers\pv_module\demand_yearly_variation.csv
